Para clasificar e identificar sátira en textos en español, puedes extraer características lingüísticas y estilísticas que capturen la ironía, el sarcasmo y el humor característicos de este tipo de discurso

Características gramaticales y de sentimiento en textos satíricos
Los textos satíricos suelen presentar un conjunto de características lingüísticas distintivas. Basándonos en análisis previos en Procesamiento de Lenguaje Natural (PLN), podemos dividirlas en dos categorías principales:

1. Características gramaticales
Mayor uso de adjetivos 📌
Se utilizan para exagerar cualidades, generar ironía o burla.
Ejemplo: "El increíble, maravilloso y completamente absurdo plan de gobierno".
Uso frecuente de sustantivos abstractos 🎭
Para referirse a conceptos generales de manera irónica.
Ejemplo: "La democracia perfecta que nadie pidió".
Más verbos en subjuntivo y condicional 🔄
Se usan para expresar hipótesis, sarcasmo o ironía.
Ejemplo: "Si los políticos fueran honestos, el mundo sería un paraíso".
Uso de pronombres personales y posesivos 🤹‍♂️
Da un tono subjetivo o coloquial al texto.
Ejemplo: "Nos prometieron el oro y el moro, pero nos dejaron con deudas".
Oraciones más largas y complejas 🏗️
Se emplean estructuras subordinadas para aumentar el dramatismo.
2. Características de sentimiento
Alto nivel de polaridad (positiva o negativa extrema) 🌡️
Suele haber una exageración en los términos empleados.
Ejemplo: "El nuevo impuesto es la mejor idea del siglo, dijeron nunca".
Alto uso de términos negativos con connotaciones sarcásticas 😏
Expresiones como "qué gran idea", "maravilloso desastre".
Presencia de emoción mezclada (positivo + negativo en una oración) 🔀
Contradicciones intencionadas para generar ironía.
Ejemplo: "El discurso estuvo tan emocionante como una siesta".
Mayor uso de palabras asociadas al humor y la burla 😂
Uso de términos como "ridículo", "esperpento", "absurdo".

In [ ]:
# The first step is download the required libraries
#!pip install librosa
#!pip install pandas
#!pip install -U scikit-learn

In [ ]:
#!pip install stanza

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 25.0 MB/s eta 0:00:00


In [ ]:
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.3/105.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.4/939.4 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 35.8 MB/s eta 0:00:00


In [ ]:
 # Load the required libraries
import pandas as pd
import numpy as np
import re

from tqdm import tqdm
tqdm.pandas()

import textstat
from collections import Counter
import spacy
import unicodedata
import emoji
import math

from textblob import TextBlob

import nltk
from nltk import Text
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.probability import FreqDist
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

from analisisFraseologico import *

from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from transformers import pipeline


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:

# Descargar WordNet y la base de datos multilingüe
nltk.download('omw-1.4')  # Base de datos multilingüe de WordNet

[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:

import pandas as pd
import numpy as np
import nltk
import re
import requests

# Descargar recursos de NLTK (solo primera vez)
nltk.download(['punkt', 'stopwords', 'wordnet', 'omw-1.4'])

# Importar después de descargar
from nltk.corpus import wordnet, stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.util import ngrams
from collections import Counter

# Configurar stopwords
stop_words_nltk = set(stopwords.words('spanish'))

# Opcional: Si necesitas WordNet en español con wn
try:
    import wn
    wn.download('omw-1.4')  # Descarga Open Multilingual WordNet
except ImportError:
    print("Paquete 'wn' no instalado. Usando solo NLTK WordNet.")

Paquete 'wn' no instalado. Usando solo NLTK WordNet.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
# Download training and test audios and CSV files
# Please, replace %PASTE_YOUR_ID_HERE% with the API key provided.
#!wget https://pln.inf.um.es/corpora/satispeech/2025/dataset/download/dev?api_key=[%INSERT_API%] -O segments.zip
#!wget https://pln.inf.um.es/corpora/satispeech/2025/SatiSPeech_phase_1_train_codalab.csv -O SatiSPeech_phase_1_train_codalab.csv
#!wget https://pln.inf.um.es/corpora/satispeech/2025/SatiSPeech_phase_1_test_codalab.csv -O SatiSPeech_phase_1_test_codalab.csv

In [ ]:
path = "/content/drive/MyDrive/Titulacion/Datasets/"

In [ ]:
# Read the CSV files
#df_train = pd.read_csv(path + "SatiSPeech_phase_1_train_codalab.csv")
#df_test = pd.read_csv(path + "SatiSPeech_phase_1_test_codalab.csv")
df_train2 = pd.read_csv(path + "SatiSPeech_phase_2_train_public.csv")
df_test2 = pd.read_csv(path + "SatiSPeech_phase_1_test_codalab.csv")


In [ ]:
df_train2

,id,label,transcription
0,5eef381b-7c3102ad.mp3,satire,"Yo creo que ya lo dice la propia frase, es pri..."
1,3db0b886-434a22fd.mp3,no-satire,"El presidente de Estados Unidos, Barack Obama,..."
2,1e3fd1a7-dfd534d8.mp3,satire,El presidente Andrés Manuel López Obrador visi...
3,2f593981-4087fa79.mp3,satire,"La sedición, porque inflación puede haber en c..."
4,815c0b94-1002ecaa.mp3,no-satire,Frenar la escalada de violencia en Gaza es una...
...,...,...,...
5995,3982793b-e3c0ec4a.mp3,no-satire,Pero la oficina del presidente francés dice qu...
5996,f0442261-0348093a.mp3,satire,No sé cómo se dice en castellano. Está siendo ...
5997,64ea7bec-c46cfd71.mp3,no-satire,"Sí, porque Rajoy dice que la razón de que esa ..."
5998,59a76f7b-75cc8d14.mp3,no-satire,Francisco era conocido como Jimmy entre los se...


In [ ]:
df_test2

,id,label,transcription
0,2abcfbec-b59cdc92.mp3,no-satire,"Para algunos deportados, mudarse a dormir aquí..."
1,9c3f7d29-727fa523.mp3,satire,Pues solo cuando me subí a un escenario a hace...
2,0e415924-8b600d71.mp3,no-satire,Se trata de los máximos responsables de la ban...
3,1f09022d-81190f9b.mp3,no-satire,En la maniobra final de aterrizaje algo se com...
4,aa352c05-532b33cc.mp3,satire,manejar el hate y digo porque por lo general e...
...,...,...,...
91,3826f75b-ba14c8e5.mp3,satire,en Mojama y fui a verlo porque a mí me dicen h...
92,86ae0b91-e42d0ad0.mp3,satire,"Y ya al volver, o sea, todo ha ido bien, y ya ..."
93,4bfde41f-989b722b.mp3,satire,Ayer me las cortaron. Porque sabías que venías...
94,53b63159-64f00c13.mp3,no-satire,El niño británico afectado con un tumor cerebr...


In [ ]:
# Codificar label
#df_train['label'] = df_train['label'].map({'no-satire': 0, 'satire': 1})
df_test2['label'] = df_test2['label'].map({'no-satire': 0, 'satire': 1})
df_train2['label'] = df_train2['label'].map({'no-satire': 0, 'satire': 1})


In [ ]:
df_train2

,id,label,transcription
0,5eef381b-7c3102ad.mp3,1,"Yo creo que ya lo dice la propia frase, es pri..."
1,3db0b886-434a22fd.mp3,0,"El presidente de Estados Unidos, Barack Obama,..."
2,1e3fd1a7-dfd534d8.mp3,1,El presidente Andrés Manuel López Obrador visi...
3,2f593981-4087fa79.mp3,1,"La sedición, porque inflación puede haber en c..."
4,815c0b94-1002ecaa.mp3,0,Frenar la escalada de violencia en Gaza es una...
...,...,...,...
5995,3982793b-e3c0ec4a.mp3,0,Pero la oficina del presidente francés dice qu...
5996,f0442261-0348093a.mp3,1,No sé cómo se dice en castellano. Está siendo ...
5997,64ea7bec-c46cfd71.mp3,0,"Sí, porque Rajoy dice que la razón de que esa ..."
5998,59a76f7b-75cc8d14.mp3,0,Francisco era conocido como Jimmy entre los se...


In [ ]:
df_train2["label"].value_counts()

,count
label,
0,3168
1,2832


In [ ]:
#separar las promeras 20 filas de df_train

#df_train = df_train.head(20)
#df_train2 = df_train2.iloc[:50]


In [ ]:
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 97.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# cargar .csv en español como una lista

# Ruta al archivo CSV en Google Drive
file_path = "/content/drive/MyDrive/Titulacion/Datasets/adverbios_conectores_satira_expandido.csv"

# Cargar el archivo CSV en un DataFrame de pandas
df = pd.read_csv(file_path, sep=';')

def normalize_text(text):
  text = text.lower()
  # Normalize the text to remove accents
  text = ''.join(c for c in unicodedata.normalize('NFD', text)
                 if unicodedata.category(c) != 'Mn')
  # Remove punctuation
  text = ''.join(c for c in text if not c in '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~')
  #text = ''.join(c for c in text if not c in '^a-zA-Záéíóúüñ¿?¡!.,;')
  return text

df['PALABRAS'] = df['PALABRAS'].apply(normalize_text)

# Convertir el DataFrame a una lista de listas
satirical_words = df['PALABRAS'].values.tolist()

# Imprimir la lista (o procesarla como necesites)
satirical_words



['a fin de cuentas',
 'absolutamente',
 'absurdo',
 'ah si',
 'ahora resulta que',
 'aparentemente',
 'asi estamos',
 'asombrosamente',
 'asombroso',
 'asquerosamente',
 'aunque parezca increible',
 'brillante',
 'casi siempre',
 'ciertamente',
 'claramente',
 'claro',
 'claro que si',
 'como bien sabemos',
 'como era de esperarse',
 'como si',
 'como siempre',
 'como todo esta bajo control',
 'completamente',
 'convenientemente',
 'curiosamente',
 'dado que todo funciona tan bien',
 'de forma exagerada',
 'de forma innecesaria',
 'de manera sorprendente',
 'de modo que',
 'de todas formas',
 'de todos modos',
 'desmesurado',
 'el mejor del mundo',
 'el peor de la historia',
 'en cuanto a eso',
 'en este caso',
 'en este contexto',
 'en fin',
 'en realidad',
 'en resumen',
 'en serio',
 'en teoria',
 'en un mundo ideal',
 'en vez de',
 'enormemente',
 'esplendido',
 'estoy impactado',
 'evidentemente',
 'exactamente',
 'excelente',
 'excepcionalmente',
 'excesivamente',
 'exorbitanteme

In [ ]:
# Descargar stopwords de NLTK en español
stopwords_es = set(stopwords.words('spanish')) #Descarga palabras en español en un listado de palabras.

# Cargar modelo de spaCy en español
nlp = spacy.load("es_core_news_sm")

# Función para normalizar texto
def preprocess_text(text):
    # 1️⃣ Convertir a minúsculas
    text = text.lower()
    # 2️⃣ Eliminar URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    # 3️⃣ Eliminar emojis
    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r'@\w+', '', text) # Eliminar menciones de usuario (@usuario)
    text = re.sub(r'#\w+', '', text)  # Eliminar hashtags
    text = re.sub(r'\d+', '', text) # Eliminar números
    text = re.sub(r'\s+', ' ', text).strip() # Eliminar espacios en blanco extra
    # 4️⃣ Eliminar caracteres especiales (manteniendo puntuación relevante)
    text = re.sub(r"[^a-zA-Záéíóúüñ¿?¡!.,;]", " ", text)
    # 6️⃣ Tokenización y lematización con spaCy
    doc = nlp(text)
    text = " ".join([token.lemma_ for token in doc if token.text not in stopwords_es])

    return text
     #texto = re.sub(r'[\U00010000-\U0010ffff]', '', text)  # Eliminar emogis
    #texto = re.sub(r'\b(?:https?://|www\.)\S+\b', '', text)  # Eliminar enlaces
     #text = " ".join([token.lemma_ for token in doc if token.is_alpha and (token.text in satirical_words or token.text.lower() not in stopwords_es)])

In [ ]:
stopwords_es

{'a',
 'al',
 'algo',
 'algunas',
 'algunos',
 'ante',
 'antes',
 'como',
 'con',
 'contra',
 'cual',
 'cuando',
 'de',
 'del',
 'desde',
 'donde',
 'durante',
 'e',
 'el',
 'ella',
 'ellas',
 'ellos',
 'en',
 'entre',
 'era',
 'erais',
 'eran',
 'eras',
 'eres',
 'es',
 'esa',
 'esas',
 'ese',
 'eso',
 'esos',
 'esta',
 'estaba',
 'estabais',
 'estaban',
 'estabas',
 'estad',
 'estada',
 'estadas',
 'estado',
 'estados',
 'estamos',
 'estando',
 'estar',
 'estaremos',
 'estará',
 'estarán',
 'estarás',
 'estaré',
 'estaréis',
 'estaría',
 'estaríais',
 'estaríamos',
 'estarían',
 'estarías',
 'estas',
 'este',
 'estemos',
 'esto',
 'estos',
 'estoy',
 'estuve',
 'estuviera',
 'estuvierais',
 'estuvieran',
 'estuvieras',
 'estuvieron',
 'estuviese',
 'estuvieseis',
 'estuviesen',
 'estuvieses',
 'estuvimos',
 'estuviste',
 'estuvisteis',
 'estuviéramos',
 'estuviésemos',
 'estuvo',
 'está',
 'estábamos',
 'estáis',
 'están',
 'estás',
 'esté',
 'estéis',
 'estén',
 'estés',
 'fue',
 'f

In [ ]:
# 🔹 Ejemplo de uso
textos = [
    "¡Hola! ¿Cómo estás? Esto cuesta $20. 😀🔥 #Ejemplo",
    "¡Claro! mamá, corazón, www.cnn.com Porque seguro que subir impuestos nos hará más ricos. 😂",
    "Magnífico, maravilloso, @cespinr estoy muy feliz",
    "El gobierno ha demostrado una eficiencia increíble... para destruir la economía. 😅 #sarcasmo",
    "Como si cambiar la bandera del país resolviera los problemas de la gente. 🤡"
]

textos_procesados = [preprocess_text(t) for t in textos]

# Mostrar resultados
for original, procesado in zip(textos, textos_procesados):
    print(f"Original: {original}\nProcesado: {procesado}\n")

Original: ¡Hola! ¿Cómo estás? Esto cuesta $20. 😀🔥 #Ejemplo
Procesado: ¡ hola ! ¿ cómo ? costar   .

Original: ¡Claro! mamá, corazón, www.cnn.com Porque seguro que subir impuestos nos hará más ricos. 😂
Procesado: ¡ claro ! mamá , corazón , seguro subir impuesto hacer rico .

Original: Magnífico, maravilloso, @cespinr estoy muy feliz
Procesado: magnífico , maravilloso , feliz

Original: El gobierno ha demostrado una eficiencia increíble... para destruir la economía. 😅 #sarcasmo
Procesado: gobierno demostrar eficiencia increíble ... destruir economía .

Original: Como si cambiar la bandera del país resolviera los problemas de la gente. 🤡
Procesado: si cambiar bandera país resolver problema gente .



In [ ]:
# aplicar preprocess_text a df_train["transcription"], con barra de progreso

# Aplicar la función preprocess_text a la columna "transcription" con tqdm
#df_train["transcription_processed"] = df_train["transcription"].progress_apply(preprocess_text)
df_test2["transcription_processed"] = df_test2["transcription"].progress_apply(preprocess_text)
df_train2["transcription_processed"] = df_train2["transcription"].progress_apply(preprocess_text)


100%|██████████| 6000/6000 [01:43<00:00, 57.90it/s]


In [ ]:
df_train2

,id,label,transcription,transcription_processed
0,5eef381b-7c3102ad.mp3,1,"Yo creo que ya lo dice la propia frase, es pri...","creer decir propio frase , privar , cada hacer..."
1,3db0b886-434a22fd.mp3,0,"El presidente de Estados Unidos, Barack Obama,...","presidente unidos , barack obama , prometer añ..."
2,1e3fd1a7-dfd534d8.mp3,1,El presidente Andrés Manuel López Obrador visi...,presidente andrés manuel lópez obrador visitar...
3,2f593981-4087fa79.mp3,1,"La sedición, porque inflación puede haber en c...","sedición , inflación poder haber cualquiera mo..."
4,815c0b94-1002ecaa.mp3,0,Frenar la escalada de violencia en Gaza es una...,frenar escalada violencia gaza prioridad . min...
...,...,...,...,...
5995,3982793b-e3c0ec4a.mp3,0,Pero la oficina del presidente francés dice qu...,oficina presidente francés decir espíritu inic...
5996,f0442261-0348093a.mp3,1,No sé cómo se dice en castellano. Está siendo ...,"saber cómo decir castellano . ser fucking , fu..."
5997,64ea7bec-c46cfd71.mp3,0,"Sí, porque Rajoy dice que la razón de que esa ...",", rajoy decir razón calificación crediticio ca..."
5998,59a76f7b-75cc8d14.mp3,0,Francisco era conocido como Jimmy entre los se...,francisco conocer jimmy seguidor deportivo . a...


In [ ]:
df_test2

,id,label,transcription,transcription_processed
0,2abcfbec-b59cdc92.mp3,0,"Para algunos deportados, mudarse a dormir aquí...","deportado , mudar él dormir aquí último recurs..."
1,9c3f7d29-727fa523.mp3,1,Pues solo cuando me subí a un escenario a hace...,pues solo subir escenario hacer monólogo viven...
2,0e415924-8b600d71.mp3,0,Se trata de los máximos responsables de la ban...,"tratar máximo responsable banda terrorista , i..."
3,1f09022d-81190f9b.mp3,0,En la maniobra final de aterrizaje algo se com...,maniobra final aterrizaje complicar ultraliger...
4,aa352c05-532b33cc.mp3,1,manejar el hate y digo porque por lo general e...,manejar hate decir general odiar bien pendejo ...
...,...,...,...,...
91,3826f75b-ba14c8e5.mp3,1,en Mojama y fui a verlo porque a mí me dicen h...,mojama ver él decir mojama ir directamente
92,86ae0b91-e42d0ad0.mp3,1,"Y ya al volver, o sea, todo ha ido bien, y ya ...","volver , , ir bien , volver , agredir aeropuer..."
93,4bfde41f-989b722b.mp3,1,Ayer me las cortaron. Porque sabías que venías...,"ayer cortar . sabía veniar tele yo , cariño . ..."
94,53b63159-64f00c13.mp3,0,El niño británico afectado con un tumor cerebr...,niño británico afectado tumor cerebral padre l...


In [ ]:
def satira(text):
  doc = nlp(text)

  text = text.lower()

  # Normalize the text to remove accents
  text = ''.join(c for c in unicodedata.normalize('NFD', text)
                 if unicodedata.category(c) != 'Mn')

  # Remove punctuation.
  text = ''.join(c for c in text if not c in '"#$%&\'()*+-/:<=>@[\\]^_`{|}~')

  satire_words_count = {}
  total_satire_words = 0
  for phrase in satirical_words:
    # Count occurrences of each phrase in the text.
    satire_words_count[phrase] = text.count(phrase.lower())  # Ensure case-insensitive comparison
    total_satire_words += satire_words_count[phrase]

  #Relación entre conectores y longitud del texto
  #Proporción de conectores con respecto al total de palabras para obtener una medida relativa de la "densidad" de conectores en el texto.
  word_count = len([token.text for token in doc if not token.is_punct])  # Número total de palabras
  satire_words_density = total_satire_words / word_count if word_count else 0

  return {
      **satire_words_count,
      "total_satire_words" : total_satire_words,
      "satire_words_density" : satire_words_density
  }

In [ ]:
#nlp = spacy.load("es_core_news_sm")  # Modelo de spaCy en español
sia = SentimentIntensityAnalyzer()   # Analizador de sentimiento de NLTK

# Cargamos el modelo de Hugging Face para detectar ironía
irony_detector = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-irony")

# Función para extraer características
def extract_features(text):
    doc = nlp(text)

    # 1️⃣ Longitud del texto
    num_words = len(word_tokenize(text))
    num_chars = len(text)

    # 2️⃣ Uso de puntuación irónica
    exclamations = text.count("!")
    #questions = text.count("?") #ojo con rhetorical_questions
    uppercase_ratio = sum(1 for c in text if c.isupper()) / max(1, len(text))  # Proporción de mayúsculas

    # 3️⃣ Conteo de palabras clave de sátira
    words = [token.text.lower() for token in doc]

    # 4️⃣ Análisis de polaridad y subjetividad
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity  # Valores entre -1 (negativo) y 1 (positivo)
    subjectivity = blob.sentiment.subjectivity  # 0 (objetivo) a 1 (subjetivo)
    # Polaridad con VADER (otro método para comparación)
    vader_polarity = sia.polarity_scores(text)['compound'] #Útil para analizar textos cortos con lenguaje coloquial.

    # 5️⃣ Análisis de ironía con modelo preentrenado
    irony_score = irony_detector(text)[0]['score']

    # 6️⃣ Conteo de adverbios, conectores y preguntas retóricas
    prop_ADV = sum(1 for token in doc if token.pos_ == "ADV") / num_words if num_words > 0 else 0
    prop_NOUN = sum(1 for token in doc if token.pos_ == "NOUN") / num_words if num_words > 0 else 0
    prop_VERB = sum(1 for token in doc if token.pos_ == "VERB") / num_words if num_words > 0 else 0
    prop_ADJ = sum(1 for token in doc if token.pos_ == "ADJ") / num_words if num_words > 0 else 0
    rhetorical_questions = sum(1 for sent in doc.sents if sent.text.strip().endswith("?"))

    # 7️⃣ Conteo de metáforas (simplificado con "como" o "es como")
    metaphors = len(re.findall(r'\bcomo\b|\bes como\b', text, re.IGNORECASE))

    satire_words_count = satira(text)  # Call the satira function
    total_satire_words = satire_words_count["total_satire_words"]
    satire_words_density = satire_words_count["satire_words_density"]

    return {
        "num_words": num_words,
        "num_chars": num_chars,
        "exclamations": exclamations,
        #"questions": questions,
        "uppercase_ratio": uppercase_ratio,
        "polarity": polarity,
        "subjectivity": subjectivity,
        "Polaridad_VADER": vader_polarity,
        "irony_score": irony_score,
        "prop_ADV" : prop_ADV,
        "prop_NOUN" : prop_NOUN,
        "prop_VERB": prop_VERB,
        "prop_ADJ": prop_ADJ,
        "rhetorical_questions": rhetorical_questions,
        "metaphors": metaphors,
        "satire_words_density" : satire_words_density,
        **satire_words_count,
        "total_satire_words" : total_satire_words
    }



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/705 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
# Ejemplo de uso con un dataset de textos satíricos
data = {
    "text": [
        "¡Claro! Porque seguro que subir impuestos nos hará más ricos.",
        "perfecto, Magnífico, maravilloso, según parece estoy muy feliz",
        "obviamente, perfecto, ni modo, no faltaba mas",
        "estoy muy feliz?",
        "El gobierno ha demostrado una eficiencia increíble... para destruir la economía.",
        "Como si cambiar la bandera del país resolviera los problemas de la gente.",
        "Me encanta este lugar, es increíblemente hermoso.",
        "Odio este clima, siempre está lloviendo.",
        "La comida estaba bien, nada fuera de lo normal.",
        "Este producto es un engaño total, no lo recomiendo.",
        "Qué maravilla de servicio, estoy muy satisfecho.",
        "estoy tan contento, que me mataría"
    ]
}

df = pd.DataFrame(data)




In [ ]:
# Extraer conectores de satira para cada texto
df_con = df["text"].apply(lambda x: pd.Series(satira(x)))

# Combinar dataset original con características extraídas
df_finalcon = pd.concat([df, df_con], axis=1)

# Mostrar resultados
print(df_finalcon)

                                                 text  a fin de cuentas  \
0   ¡Claro! Porque seguro que subir impuestos nos ...               0.0   
1   perfecto, Magnífico, maravilloso, según parece...               0.0   
2       obviamente, perfecto, ni modo, no faltaba mas               0.0   
3                                    estoy muy feliz?               0.0   
4   El gobierno ha demostrado una eficiencia incre...               0.0   
5   Como si cambiar la bandera del país resolviera...               0.0   
6   Me encanta este lugar, es increíblemente hermoso.               0.0   
7            Odio este clima, siempre está lloviendo.               0.0   
8     La comida estaba bien, nada fuera de lo normal.               0.0   
9   Este producto es un engaño total, no lo recomi...               0.0   
10   Qué maravilla de servicio, estoy muy satisfecho.               0.0   
11                 estoy tan contento, que me mataría               0.0   

    absolutamente  absur

In [ ]:
df_finalcon["segun parece"].value_counts()

,count
segun parece,
0.0,11
1.0,1


In [ ]:
# Extraer características para cada texto
df_features = df["text"].apply(lambda x: pd.Series(extract_features(x)))
#df_features = df["text"].apply(lambda x: pd.Series(otra_features(x)))
#extract_additional_features= df["text"].apply(lambda x: pd.Series(extract_additional_features(x)))

# Combinar dataset original con características extraídas
df_final = pd.concat([df, df_features], axis=1) #,extract_additional_features

# Mostrar resultados
print(df_final)

                                                 text  num_words  num_chars  \
0   ¡Claro! Porque seguro que subir impuestos nos ...       12.0       61.0   
1   perfecto, Magnífico, maravilloso, según parece...       11.0       62.0   
2       obviamente, perfecto, ni modo, no faltaba mas       10.0       45.0   
3                                    estoy muy feliz?        4.0       16.0   
4   El gobierno ha demostrado una eficiencia incre...       13.0       80.0   
5   Como si cambiar la bandera del país resolviera...       14.0       73.0   
6   Me encanta este lugar, es increíblemente hermoso.        9.0       49.0   
7            Odio este clima, siempre está lloviendo.        8.0       40.0   
8     La comida estaba bien, nada fuera de lo normal.       11.0       47.0   
9   Este producto es un engaño total, no lo recomi...       11.0       51.0   
10   Qué maravilla de servicio, estoy muy satisfecho.        9.0       48.0   
11                 estoy tan contento, que me matarí

In [ ]:
df_final

,text,num_words,num_chars,exclamations,uppercase_ratio,polarity,subjectivity,Polaridad_VADER,irony_score,prop_ADV,...,totalmente,tremendamente,tremendo,una locura,una maravilla,y asi nos va,y claro,y yo soy el papa,ya que estamos,total_satire_words
0,¡Claro! Porque seguro que subir impuestos nos ...,12.0,61.0,1.0,0.032787,0.00,0.00,0.0000,0.643576,0.250000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
1,"perfecto, Magnífico, maravilloso, según parece...",11.0,62.0,0.0,0.016129,0.00,0.00,0.3182,0.923144,0.090909,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
2,"obviamente, perfecto, ni modo, no faltaba mas",10.0,45.0,0.0,0.000000,0.00,0.00,0.0258,0.899620,0.200000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
3,estoy muy feliz?,4.0,16.0,0.0,0.000000,0.00,0.00,0.0000,0.940350,0.250000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,El gobierno ha demostrado una eficiencia incre...,13.0,80.0,0.0,0.012500,0.00,0.00,0.3400,0.858387,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
5,Como si cambiar la bandera del país resolviera...,14.0,73.0,0.0,0.013699,0.00,0.00,0.0000,0.511303,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
6,"Me encanta este lugar, es increíblemente hermoso.",9.0,49.0,0.0,0.020408,0.00,0.00,0.0000,0.810739,0.111111,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
7,"Odio este clima, siempre está lloviendo.",8.0,40.0,0.0,0.025000,0.00,0.00,0.0000,0.771237,0.125000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,"La comida estaba bien, nada fuera de lo normal.",11.0,47.0,0.0,0.021277,0.15,0.65,0.0000,0.733582,0.181818,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,"Este producto es un engaño total, no lo recomi...",11.0,51.0,0.0,0.019608,0.00,0.75,-0.2960,0.857262,0.090909,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Aplicar la función extract_features a la columna "transcription_processed" con tqdm
#df_train_features = df_train["transcription_processed"].progress_apply(lambda x: pd.Series(extract_features(x)))
df_test2_features = df_test2["transcription_processed"].progress_apply(lambda x: pd.Series(extract_features(x)))
df_train2_features = df_train2["transcription_processed"].progress_apply(lambda x: pd.Series(extract_features(x)))

100%|██████████| 6000/6000 [40:39<00:00,  2.46it/s]


In [ ]:
#fra_train = resultadoFraseologico(df_train)
#fra_test = resultadoFraseologico(df_test)
fra_train2 = resultadoFraseologico(df_train2)
fra_test2 = resultadoFraseologico(df_test2)

In [ ]:
fra_train2

,MeanWordLen,LexicalDiversity,MeanSentenceLen,StdevSentenceLen,MeanParagraphLen,DocumentLen,WordsPerText,SentencesPerText,MeanDifferenceSentenceLengths
0,4.761905,85.714286,29.000000,0.000000,29.0,136,21,1,0.000000
1,6.192308,100.000000,15.500000,1.500000,31.0,195,26,2,-3.000000
2,6.413793,86.206897,16.000000,5.000000,32.0,219,29,2,-10.000000
3,6.615385,84.615385,16.500000,3.500000,33.0,210,26,2,7.000000
4,6.838710,96.774194,8.750000,3.897114,35.0,247,31,4,-0.333333
...,...,...,...,...,...,...,...,...,...
5995,6.684211,94.736842,14.333333,7.133645,43.0,299,38,3,6.000000
5996,5.736842,84.210526,6.400000,2.154066,32.0,151,19,5,1.000000
5997,7.435897,89.743590,11.750000,2.861381,47.0,341,39,4,0.666667
5998,6.500000,96.875000,6.500000,2.061553,39.0,248,32,6,-0.600000


In [ ]:
def calculate_dependency_metrics(text):
  #nlp = spacy.load("es_core_news_sm")
  doc = nlp(text)

  total_depth = 0
  total_length = 0
  sentence_count = 0

  results = []

  for sent in doc.sents:
    depths = [token.head.i - token.i if token.head != token else 0 for token in sent]
    depth = max(depths) if depths else 0  # Profundidad máxima en la oración
    length = sum(abs(dep) for dep in depths)  # Longitud de dependencia total

    total_depth += depth
    total_length += length
    sentence_count += 1

    results.append({"sentence": sent.text, "depth": depth, "length": length})

  avg_depth = total_depth / sentence_count if sentence_count else 0
  avg_length = total_length / sentence_count if sentence_count else 0

  return {
      "avg_depth": avg_depth,
      "avg_length": avg_length
      #"sentence_metrics": results
  }


In [ ]:
# Texto de ejemplo
text = "El perro que vi ayer en el parque estaba jugando con una pelota. Era muy amigable."
metrics = calculate_dependency_metrics(text)

print("Profundidad sintáctica promedio:", metrics["avg_depth"])
print("Longitud de dependencia promedio:", metrics["avg_length"])
#print("\nDetalles por oración:")
#for item in metrics["sentence_metrics"]:
#    print(f"- {item['sentence']}\n  Profundidad: {item['depth']}, Longitud: {item['length']}")

Profundidad sintáctica promedio: 5.0
Longitud de dependencia promedio: 17.5


In [ ]:
# aplicar calculate_dependency_metrics a df_train2["transcription_processed"]
#df_train_dependency= df_train["transcription_processed"].progress_apply(lambda x: pd.Series(calculate_dependency_metrics(x)))
#df_test_dependency = df_test["transcription_processed"].progress_apply(lambda x: pd.Series(calculate_dependency_metrics(x)))
df_train2_dependency = df_train2["transcription_processed"].progress_apply(lambda x: pd.Series(calculate_dependency_metrics(x)))
df_test2_dependency = df_test2["transcription_processed"].progress_apply(lambda x: pd.Series(calculate_dependency_metrics(x)))



100%|██████████| 96/96 [00:01<00:00, 93.07it/s]


In [ ]:
df_train2_dependency

,avg_depth,avg_length
0,2.000000,86.000000
1,3.500000,41.500000
2,3.000000,38.000000
3,1.000000,52.000000
4,2.250000,17.250000
...,...,...
5995,2.000000,43.333333
5996,2.000000,17.250000
5997,2.750000,27.750000
5998,0.833333,11.166667


In [ ]:
#creo una lista de palabras comunes en español en base a CREA
file_path = "/content/drive/MyDrive/Titulacion/Datasets/PalabrasComunes/CREA_PalabrasComunes.txt"

In [ ]:
df_CREA = pd.read_csv(file_path, sep='\t', encoding='latin-1')

In [ ]:
df_CREA

,Orden,Palabra,Frec.absoluta,Frec.normalizada
0,1,de,"9,999,518",6554555
1,2,la,"6,277,560",4114859
2,3,que,"4,681,839",3068885
3,4,el,"4,569,652",2995348
4,5,en,"4,234,281",2775516
...,...,...,...,...
9995,9996,for,"1,182",774
9996,9997,fuiste,"1,182",774
9997,9998,galileo,"1,182",774
9998,9999,juanito,"1,182",774


In [ ]:
df_CREA['Palabra'] = df_CREA['Palabra'].apply(normalize_text)
df_CREA['Palabra']

,Palabra
0,de
1,la
2,que
3,el
4,en
...,...
9995,for
9996,fuiste
9997,galileo
9998,juanito


In [ ]:
# prompt: crear un set con las palabras de la df_CREA["Palabra"]

common_words_es = set(df_CREA["Palabra"])
common_words_es



{'pollo',
 'articular',
 'compania ',
 'extremadamente  ',
 'pecados ',
 'crimenes ',
 'meta ',
 'hablado ',
 'tortura',
 'sintomas ',
 'danos ',
 'buena ',
 'deseable ',
 'esperan ',
 'penal',
 'numeros ',
 'hermoso ',
 'temo ',
 'alumnos ',
 'tinta',
 'movio ',
 'virus',
 'accidente',
 'claridad',
 'liderazgo',
 'realiza',
 'mismos  ',
 'beber',
 'protector',
 'honduras ',
 'representar',
 'paraiso',
 'conscientes ',
 'situada',
 'humor ',
 'golpes',
 'conductor ',
 'peligrosa',
 'siguen',
 'polemica ',
 'agonia',
 'infraestructura',
 'republicano',
 'abren',
 'doscientos ',
 'esperanzas ',
 'entrega',
 'pies',
 'aplicada',
 'denomina ',
 'hong ',
 'electores',
 'aroma ',
 'ensena ',
 'luisa',
 'desechos ',
 'pico',
 'unirse',
 'realizarse',
 'sentimos ',
 'completo ',
 'paradojicamente ',
 'camioneta ',
 'rostros',
 'gobernacion ',
 'policia',
 'reyes',
 'fujimori',
 'noticia',
 'destaco ',
 'contribuye',
 'ja',
 'gritaba',
 'yacimientos ',
 'genes ',
 'recetas',
 'dira',
 'discusio

In [ ]:
def flesch_score(text, lang="es"):
    """Calcula el puntaje de Flesch."""
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    if not sentences:
        return None  # Evita errores si el texto está vacío

    num_sentences = len(sentences)
    num_words = sum(len(s.split()) for s in sentences)
    num_syllables = sum(textstat.syllable_count(word) for word in text.split())

    asl = num_words / num_sentences  # Longitud media de oraciones
    asw = num_syllables / num_words  # Sílabas promedio por palabra

    score = 206.84 - (1.02 * asl) - (60 * asw)  #PARA TEXTO EN ESPAÑOL
    #score = 206.835 - (1.015 * asl) - (84.6 * asw)  #PARA TEXTO EN INGLÉS

    return round(score, 2)


In [ ]:
def lexical_entropy(text):
    """Calcula la entropía léxica (variedad de palabras)."""
    words = text.lower().split()
    word_counts = Counter(words)
    total_words = sum(word_counts.values())
    entropy = -sum((count / total_words) * math.log2(count / total_words) for count in word_counts.values())
    return round(entropy, 2)

In [ ]:
def syntactic_pattern_repetition(text):
    """Calcula la repetición de patrones sintácticos."""
    #nlp = nlp_es if lang == "es" else nlp_en
    doc = nlp(text)

    dep_patterns = [token.dep_ for token in doc]
    dep_counts = Counter(dep_patterns)

    # Índice de repetición: cuántas veces se repiten las dependencias sintácticas más frecuentes
    repetition_index = max(dep_counts.values()) / len(dep_patterns) if dep_patterns else 0
    return round(repetition_index, 2)

In [ ]:
def unusual_word_frequency(text):
    """Calcula la frecuencia de palabras inusuales comparadas con un corpus de palabras comunes."""
    words_in_text = set(word.lower() for word in text.split())
    common_set = common_words_es
    unusual_words = words_in_text - common_set
    return round(len(unusual_words) / len(words_in_text), 2) if words_in_text else 0

In [ ]:
def analyze_text(text):
    """Analiza el texto y calcula todas las métricas."""
    return {
        "Flesch Score": flesch_score(text),
        "Lexical Entropy": lexical_entropy(text),
        "Syntactic Repetition": syntactic_pattern_repetition(text),
        "Unusual Word Frequency": unusual_word_frequency(text)
    }

In [ ]:
# Ejemplo de uso
text = "El perro corre por el parque. Es un día soleado y agradable. La brisa fresca sopla suavemente."
results = analyze_text(text)

print("Resultados del análisis:")
for metric, value in results.items():
    print(f"{metric}: {value}")

Resultados del análisis:
Flesch Score: 116.35
Lexical Entropy: 3.97
Syntactic Repetition: 0.2
Unusual Word Frequency: 0.38


In [ ]:
df_train2_analyze_text = df_train2["transcription_processed"].progress_apply(lambda x: pd.Series(analyze_text(x)))
#df_test_analyze_text = df_test["transcription_processed"].progress_apply(lambda x: pd.Series(analyze_text(x)))
df_test2_analyze_text = df_test2["transcription_processed"].progress_apply(lambda x: pd.Series(analyze_text(x)))

100%|██████████| 96/96 [00:01<00:00, 80.34it/s]


In [ ]:
df_train2_analyze_text

,Flesch Score,Lexical Entropy,Syntactic Repetition,Unusual Word Frequency
0,107.57,3.97,0.28,0.50
1,84.46,4.74,0.26,0.50
2,65.54,4.66,0.16,0.63
3,92.32,4.39,0.21,0.62
4,61.52,4.84,0.26,0.52
...,...,...,...,...
5995,71.74,5.18,0.21,0.55
5996,121.33,4.05,0.41,0.62
5997,64.71,5.04,0.21,0.49
5998,77.59,4.84,0.23,0.52


#¿Cómo podrían ayudar las métricas anteriores?
**Puntaje de Flesch (Legibilidad)**

La sátira a veces usa oraciones más largas y complejas, reduciendo la legibilidad.

Otras veces usa frases cortas e irónicas, lo que podría generar una variabilidad inusual en la puntuación de Flesch.

**Entropía léxica (Variedad de palabras)**

La sátira suele usar un vocabulario más variado (para exagerar o ridiculizar).

Si la entropía es muy alta, podría indicar sarcasmo o sátira.

**Repetición de patrones sintácticos**

Si un texto usa estructuras gramaticales poco comunes o con muchas subordinaciones, podría ser indicativo de sátira.

Sin embargo, algunos textos satíricos imitan discursos políticos o noticias, lo que podría hacerlos parecer más convencionales en esta métrica.

**Frecuencia de palabras inusuales**

La sátira a menudo incluye términos inusuales, eufemismos, ironía o dobles sentidos.

Un alto porcentaje de palabras poco frecuentes podría indicar un uso intencionalmente rebuscado o humorístico.

In [ ]:
# Combinar el DataFrame original con las nuevas características
#df_train_final = pd.concat([df_train,  fra_train, df_train_features, df_train_dependency, df_train_analyze_text,df_train_additional], axis=1)
df_test2_final = pd.concat([df_test2, fra_test2, df_test2_features, df_test2_dependency, df_test2_analyze_text], axis=1)
df_train2_final = pd.concat([df_train2, fra_train2, df_train2_features, df_train2_dependency, df_train2_analyze_text], axis=1)#, df_train2_additional

# Mostrar el DataFrame resultante
df_train2_final

,id,label,transcription,transcription_processed,MeanWordLen,LexicalDiversity,MeanSentenceLen,StdevSentenceLen,MeanParagraphLen,DocumentLen,...,y claro,y yo soy el papa,ya que estamos,total_satire_words,avg_depth,avg_length,Flesch Score,Lexical Entropy,Syntactic Repetition,Unusual Word Frequency
0,5eef381b-7c3102ad.mp3,1,"Yo creo que ya lo dice la propia frase, es pri...","creer decir propio frase , privar , cada hacer...",4.761905,85.714286,29.000000,0.000000,29.0,136,...,0.0,0.0,0.0,0.0,2.000000,86.000000,107.57,3.97,0.28,0.50
1,3db0b886-434a22fd.mp3,0,"El presidente de Estados Unidos, Barack Obama,...","presidente unidos , barack obama , prometer añ...",6.192308,100.000000,15.500000,1.500000,31.0,195,...,0.0,0.0,0.0,0.0,3.500000,41.500000,84.46,4.74,0.26,0.50
2,1e3fd1a7-dfd534d8.mp3,1,El presidente Andrés Manuel López Obrador visi...,presidente andrés manuel lópez obrador visitar...,6.413793,86.206897,16.000000,5.000000,32.0,219,...,0.0,0.0,0.0,0.0,3.000000,38.000000,65.54,4.66,0.16,0.63
3,2f593981-4087fa79.mp3,1,"La sedición, porque inflación puede haber en c...","sedición , inflación poder haber cualquiera mo...",6.615385,84.615385,16.500000,3.500000,33.0,210,...,0.0,0.0,0.0,0.0,1.000000,52.000000,92.32,4.39,0.21,0.62
4,815c0b94-1002ecaa.mp3,0,Frenar la escalada de violencia en Gaza es una...,frenar escalada violencia gaza prioridad . min...,6.838710,96.774194,8.750000,3.897114,35.0,247,...,0.0,0.0,0.0,0.0,2.250000,17.250000,61.52,4.84,0.26,0.52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,3982793b-e3c0ec4a.mp3,0,Pero la oficina del presidente francés dice qu...,oficina presidente francés decir espíritu inic...,6.684211,94.736842,14.333333,7.133645,43.0,299,...,0.0,0.0,0.0,0.0,2.000000,43.333333,71.74,5.18,0.21,0.55
5996,f0442261-0348093a.mp3,1,No sé cómo se dice en castellano. Está siendo ...,"saber cómo decir castellano . ser fucking , fu...",5.736842,84.210526,6.400000,2.154066,32.0,151,...,0.0,0.0,0.0,2.0,2.000000,17.250000,121.33,4.05,0.41,0.62
5997,64ea7bec-c46cfd71.mp3,0,"Sí, porque Rajoy dice que la razón de que esa ...",", rajoy decir razón calificación crediticio ca...",7.435897,89.743590,11.750000,2.861381,47.0,341,...,0.0,0.0,0.0,0.0,2.750000,27.750000,64.71,5.04,0.21,0.49
5998,59a76f7b-75cc8d14.mp3,0,Francisco era conocido como Jimmy entre los se...,francisco conocer jimmy seguidor deportivo . a...,6.500000,96.875000,6.500000,2.061553,39.0,248,...,0.0,0.0,0.0,0.0,0.833333,11.166667,77.59,4.84,0.23,0.52


In [ ]:
df_train2_final["total_satire_words"].value_counts()

,count
total_satire_words,
0.0,5356
1.0,555
2.0,78
3.0,8
4.0,2
6.0,1


In [ ]:
df_test2_final

,id,label,transcription,transcription_processed,MeanWordLen,LexicalDiversity,MeanSentenceLen,StdevSentenceLen,MeanParagraphLen,DocumentLen,...,y claro,y yo soy el papa,ya que estamos,total_satire_words,avg_depth,avg_length,Flesch Score,Lexical Entropy,Syntactic Repetition,Unusual Word Frequency
0,2abcfbec-b59cdc92.mp3,0,"Para algunos deportados, mudarse a dormir aquí...","deportado , mudar él dormir aquí último recurs...",5.400000,96.666667,12.000000,2.943920,36.0,201,...,0.0,0.0,0.0,0.0,1.000000,37.333333,99.26,4.85,0.19,0.42
1,9c3f7d29-727fa523.mp3,1,Pues solo cuando me subí a un escenario a hace...,pues solo subir escenario hacer monólogo viven...,6.172414,82.758621,17.000000,9.000000,34.0,216,...,0.0,0.0,0.0,0.0,9.000000,46.000000,79.10,4.54,0.24,0.65
2,0e415924-8b600d71.mp3,0,Se trata de los máximos responsables de la ban...,"tratar máximo responsable banda terrorista , i...",6.977778,97.777778,8.142857,2.531435,57.0,376,...,0.0,0.0,0.0,1.0,2.857143,14.714286,84.35,5.39,0.21,0.62
3,1f09022d-81190f9b.mp3,0,En la maniobra final de aterrizaje algo se com...,maniobra final aterrizaje complicar ultraliger...,7.222222,100.000000,8.800000,2.785678,44.0,307,...,0.0,0.0,0.0,0.0,2.000000,20.200000,75.81,5.09,0.25,0.68
4,aa352c05-532b33cc.mp3,1,manejar el hate y digo porque por lo general e...,manejar hate decir general odiar bien pendejo ...,6.307692,100.000000,26.000000,0.000000,26.0,189,...,0.0,0.0,0.0,1.0,1.000000,113.000000,60.32,4.70,0.23,0.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,3826f75b-ba14c8e5.mp3,1,en Mojama y fui a verlo porque a mí me dicen h...,mojama ver él decir mojama ir directamente,5.142857,85.714286,7.000000,0.000000,7.0,42,...,0.0,0.0,0.0,0.0,0.000000,10.000000,71.13,2.52,0.43,0.50
92,86ae0b91-e42d0ad0.mp3,1,"Y ya al volver, o sea, todo ha ido bien, y ya ...","volver , , ir bien , volver , agredir aeropuer...",5.454545,72.727273,7.666667,7.318166,23.0,94,...,0.0,0.0,0.0,0.0,1.000000,35.000000,152.04,3.22,0.52,0.54
93,4bfde41f-989b722b.mp3,1,Ayer me las cortaron. Porque sabías que venías...,"ayer cortar . sabía veniar tele yo , cariño . ...",5.280000,92.000000,5.000000,2.329929,35.0,170,...,0.0,0.0,0.0,0.0,1.142857,8.571429,106.33,4.51,0.29,0.56
94,53b63159-64f00c13.mp3,0,El niño británico afectado con un tumor cerebr...,niño británico afectado tumor cerebral padre l...,6.366667,96.666667,11.333333,4.027682,34.0,226,...,0.0,0.0,0.0,0.0,3.000000,25.000000,70.49,4.89,0.21,0.35


In [ ]:
# prompt: en otro df, separar las caracteristicas rhetorical_questions,  metaphors, **satire_words_count, satire_words_density, total_satire_words

# Assuming df_train2_final is your DataFrame
#df_seman_ft = df_train2_final[['polarity', 'subjectivity', 'Polaridad_VADER', 'irony_score']].copy()
#df_sintac_ft = df_train2_final[['prop_ADV', 'prop_NOUN', 'prop_VERB', 'prop_ADJ']].copy()
#df_morfo_ft = df_train2_final[['num_words', 'num_chars', 'exclamations', 'uppercase_ratio', "MeanWordLen", "LexicalDiversity", "MeanSentenceLen", "StdevSentenceLen", "MeanParagraphLen", "DocumentLen", "WordsPerText", "SentencesPerText", "SentencesPerText"]].copy()
#df_df_fraseo_ft = df_train2_final[['rhetorical_questions', 'metaphors'] + list(df_train2_final.columns[df_train2_final.columns.get_loc('satire_words_density'):])]

In [ ]:
path = "/content/drive/MyDrive/Titulacion/DatasetsFinales/"

In [ ]:
#grabar los df finales

#df_train_final.to_json(path + 'df_train_feat.jsonl', orient='records', lines=True, force_ascii=False)
df_test2_final.to_json(path + 'df_test2_feat.jsonl', orient='records', lines=True, force_ascii=False)
df_train2_final.to_json(path + 'df_train2_feat.jsonl', orient='records', lines=True, force_ascii=False)


In [ ]:
import pandas as pd
import json

# 1. Cargar el dataset preprocesado
file_path = '/content/drive/MyDrive/Titulacion/DatasetsFinales/df_train2_feat.jsonl'
data = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))
df = pd.DataFrame(data)

# 2. Crear DataFrames para cada tipo de características
# Características fraseológicas
df_fraseologicas = pd.DataFrame({
    'id': df['id'],
    'num_words': df['num_words'],
    'num_chars': df['num_chars'],
    'exclamations': df['exclamations'],
    'uppercase_ratio': df['uppercase_ratio'],
    'satire_words_density': df['satire_words_density'],
    'total_satire_words': df['total_satire_words'],
    'rhetorical_questions': df['rhetorical_questions'],
    'metaphors': df['metaphors'],
    'MeanWordLen': df['MeanWordLen'],
    'StdevSentenceLen': df['StdevSentenceLen'],
    'MeanSentenceLen': df['MeanSentenceLen'],
    'WordsPerText': df['WordsPerText'],
    'SentencesPerText': df['SentencesPerText'],
    'MeanParagraphLen': df['MeanParagraphLen'],
    'StdevSentenceLen': df['StdevSentenceLen'],
    'LexicalDiversity': df['LexicalDiversity'],
    'DocumentLen': df['DocumentLen'],
    'MeanDifferenceSentenceLengths': df['MeanDifferenceSentenceLengths']
})

# Características sintácticas
df_sintacticas = pd.DataFrame({
    'id': df['id'],
    'prop_ADV': df['prop_ADV'],
    'prop_NOUN': df['prop_NOUN'],
    'prop_VERB': df['prop_VERB'],
    'prop_ADJ': df['prop_ADJ'],
    'avg_depth': df['avg_depth'],
    'avg_length': df['avg_length'],
    'Syntactic Repetition': df['Syntactic Repetition'],


})

# Características semánticas
df_semanticas = pd.DataFrame({
    'id': df['id'],
    'polarity': df['polarity'],
    'subjectivity': df['subjectivity'],
    'Polaridad_VADER': df['Polaridad_VADER'],
    'irony_score': df['irony_score'],
    'Lexical Entropy': df['Lexical Entropy'],
    'Unusual Word Frequency': df['Unusual Word Frequency'],
    'Flesch Score': df['Flesch Score'],
})

# 3. Mostrar los DataFrames como en la imagen
def display_features(df, title):
    print(f"\n{title}")
    print("-"*70)
    # Mostrar solo las primeras 5 filas y las últimas 5 filas como en la imagen
    display(pd.concat([df.head(), df.tail()]))
    print(f"\n{len(df)} rows × {len(df.columns)} columns")

# Mostrar cada conjunto de características
display_features(df_fraseologicas, "CARACTERÍSTICAS FRASEOLÓGICAS")
display_features(df_sintacticas, "CARACTERÍSTICAS SINTÁCTICAS")
display_features(df_semanticas, "CARACTERÍSTICAS SEMÁNTICAS")

# 4. Opcional: Guardar los resultados en archivos CSV
df_fraseologicas.to_csv('caracteristicas_fraseologicas.csv', index=False)
df_sintacticas.to_csv('caracteristicas_sintacticas.csv', index=False)
df_semanticas.to_csv('caracteristicas_semanticas.csv', index=False)



CARACTERÍSTICAS FRASEOLÓGICAS
----------------------------------------------------------------------


,id,num_words,num_chars,exclamations,uppercase_ratio,satire_words_density,total_satire_words,rhetorical_questions,metaphors,MeanWordLen,StdevSentenceLen,MeanSentenceLen,WordsPerText,SentencesPerText,MeanParagraphLen,LexicalDiversity,DocumentLen,MeanDifferenceSentenceLengths
0,5eef381b-7c3102ad.mp3,29.0,136.0,0.0,0.0,0.000000,0.0,0.0,0.0,4.761905,0.000000,29.000000,21,1,29.0,85.714286,136,0.000000
1,3db0b886-434a22fd.mp3,31.0,196.0,0.0,0.0,0.000000,0.0,0.0,0.0,6.192308,1.500000,15.500000,26,2,31.0,100.000000,195,-3.000000
2,1e3fd1a7-dfd534d8.mp3,32.0,220.0,0.0,0.0,0.000000,0.0,0.0,0.0,6.413793,5.000000,16.000000,29,2,32.0,86.206897,219,-10.000000
3,2f593981-4087fa79.mp3,33.0,211.0,0.0,0.0,0.000000,0.0,0.0,0.0,6.615385,3.500000,16.500000,26,2,33.0,84.615385,210,7.000000
4,815c0b94-1002ecaa.mp3,35.0,250.0,0.0,0.0,0.000000,0.0,0.0,0.0,6.838710,3.897114,8.750000,31,4,35.0,96.774194,247,-0.333333
5995,3982793b-e3c0ec4a.mp3,43.0,301.0,0.0,0.0,0.000000,0.0,0.0,0.0,6.684211,7.133645,14.333333,38,3,43.0,94.736842,299,6.000000
5996,f0442261-0348093a.mp3,32.0,155.0,0.0,0.0,0.105263,2.0,0.0,0.0,5.736842,2.154066,6.400000,19,5,32.0,84.210526,151,1.000000
5997,64ea7bec-c46cfd71.mp3,47.0,344.0,0.0,0.0,0.000000,0.0,0.0,0.0,7.435897,2.861381,11.750000,39,4,47.0,89.743590,341,0.666667
5998,59a76f7b-75cc8d14.mp3,39.0,253.0,0.0,0.0,0.000000,0.0,0.0,0.0,6.500000,2.061553,6.500000,32,6,39.0,96.875000,248,-0.600000
5999,6e6fc787-a59e99c4.mp3,54.0,281.0,0.0,0.0,0.000000,0.0,3.0,0.0,5.461538,5.131601,9.000000,39,6,54.0,89.743590,276,0.400000



6000 rows × 18 columns

CARACTERÍSTICAS SINTÁCTICAS
----------------------------------------------------------------------


,id,prop_ADV,prop_NOUN,prop_VERB,prop_ADJ,avg_depth,avg_length,Syntactic Repetition
0,5eef381b-7c3102ad.mp3,0.000000,0.137931,0.379310,0.000000,2.000000,86.000000,0.28
1,3db0b886-434a22fd.mp3,0.000000,0.322581,0.225806,0.096774,3.500000,41.500000,0.26
2,1e3fd1a7-dfd534d8.mp3,0.093750,0.218750,0.218750,0.187500,3.000000,38.000000,0.16
3,2f593981-4087fa79.mp3,0.000000,0.303030,0.151515,0.121212,1.000000,52.000000,0.21
4,815c0b94-1002ecaa.mp3,0.000000,0.257143,0.285714,0.200000,2.250000,17.250000,0.26
5995,3982793b-e3c0ec4a.mp3,0.023256,0.302326,0.279070,0.186047,2.000000,43.333333,0.21
5996,f0442261-0348093a.mp3,0.062500,0.187500,0.093750,0.062500,2.000000,17.250000,0.41
5997,64ea7bec-c46cfd71.mp3,0.021277,0.234043,0.297872,0.191489,2.750000,27.750000,0.21
5998,59a76f7b-75cc8d14.mp3,0.025641,0.282051,0.153846,0.230769,0.833333,11.166667,0.23
5999,6e6fc787-a59e99c4.mp3,0.037037,0.185185,0.203704,0.092593,2.833333,17.833333,0.28



6000 rows × 8 columns

CARACTERÍSTICAS SEMÁNTICAS
----------------------------------------------------------------------


,id,polarity,subjectivity,Polaridad_VADER,irony_score,Lexical Entropy,Unusual Word Frequency,Flesch Score
0,5eef381b-7c3102ad.mp3,0.000000,0.000000,0.0000,0.708257,3.97,0.50,107.57
1,3db0b886-434a22fd.mp3,0.033333,0.066667,0.1027,0.562723,4.74,0.50,84.46
2,1e3fd1a7-dfd534d8.mp3,0.000000,1.000000,0.0000,0.551360,4.66,0.63,65.54
3,2f593981-4087fa79.mp3,0.000000,0.000000,0.0000,0.650488,4.39,0.62,92.32
4,815c0b94-1002ecaa.mp3,0.000000,0.000000,0.0000,0.504903,4.84,0.52,61.52
5995,3982793b-e3c0ec4a.mp3,0.000000,0.000000,-0.6249,0.516391,5.18,0.55,71.74
5996,f0442261-0348093a.mp3,-0.600000,0.800000,-0.5106,0.752366,4.05,0.62,121.33
5997,64ea7bec-c46cfd71.mp3,0.600000,0.900000,0.4939,0.670326,5.04,0.49,64.71
5998,59a76f7b-75cc8d14.mp3,-0.280556,0.355556,-0.6249,0.786734,4.84,0.52,77.59
5999,6e6fc787-a59e99c4.mp3,0.000000,0.000000,0.6747,0.856700,5.06,0.51,117.43



6000 rows × 8 columns
